## 第16章 异步和并发

- **基本概念：**
    - 并行(Parallelism)：多个任务真正同时执行，需要多核/多进程支持。
    - 并发(Concurrency)：单个进程内多个任务之间快速切换注意力，每个时刻只做一件事。
        - 并发类型：
            - 线程(Threading)：抢占式多任务，由操作系统管理多任务切换，每个任务运行在一个线程中(单进程多线程)。
            - 异步(Asynchrony)：协作式多任务，由Python自身管理多任务切换，单进程单线程内管理多任务。
                - 异步不是并行，异步的价值在于——在等待IO时做别的事，而非让多件事同时跑。
        - 适应场景：不适应CPU密集型任务。
            - 处理I/O绑定任务：如网络请求、文件读写、输入输出等。
            - 提高感知响应性：即使程序在执行耗时任务，也能响应用户事件。
            - 定时任务：如每5分钟保存临时文件，无论程序其他部分在做什么。
        - 注意事项：
            - 并发并不真正加速执行，因为无论是串行还是并发，总时间不变。事实上，因切换开销，并发甚至可能**更慢**。
            - Python中的GIL(全局解释器锁)，确保任何单个Python进程只能使用一个CPU核心，因此，无论用异步还是线程，都**无法实现真正的并行**。

- **同步版Collatz猜想示例：**

In [ ]:
BOUND = 10**5

# 查找Collatz序列的计算步数
def collatz(n):
    steps = 0
    while n > 1:
        if n % 2:
            n = n * 3 + 1
        else:
            n //= 2
        steps += 1
    return steps

# 查找与目标计算步数相同的Collatz序列的个数
def length_collatz(target):
    count = 0
    for i in range(2, BOUND):
        if collatz(i) == target:
            count += 1
    return count

# 获取用户输入的整数
def get_input(prompt):
    while True:
        value = input(prompt)
        try:
            value = int(value)
        except ValueError:
            print("请输入一个整数")
            continue
        if value <= 0:
            print("请输入一个大于0的整数")
        else:
            return value

def main():
    target = get_input("请输入Collatz序列的计算步数：")
    count = length_collatz(target)
    guess = get_input("请输入你猜的Collatz序列的个数：")
    if guess == count:
        print("恭喜你猜对了！")
    else:
        print(f"很遗憾，你猜错了。实际的数是{count}")

if __name__ == "__main__":
    main()

- **原生协程(Coroutine)**：又称协程函数，可以在特定位置暂停并恢复执行的函数，以实现多任务处理。
    - 声明协程函数：
        - `async def`: 定义一个协程函数，直接调用协程函数**不会立即执行**，而是返回一个**协程对象**，协程对象必须被`await`或被事件循环调度才会执行。
    - 调用协程函数：
        - `await`：调用协程函数，这种模式是**串行**的，必须等它完成才能继续执行。记住：`await`**只能在**`async def`定义的协程函数内部使用！
        - `asyncio.create_task()`：创建一个任务对象，将协程函数**提交给事件循环独立调度**，不需要立即等待，可以继续执行其他代码。
    - 协程入口函数：
        - `asyncio.run()`：因为`await`只能在协程函数内部使用，所以需要在主程序中调用`asyncio.run()`来启动协程函数。记住：`asyncio.run()`应该**只调用一次**，作为异步程序的最外层入口。
    - 其他协程方法：
        - `asyncio.sleep()`：异步等待，暂停当前协程，事件循环可继续运行其他协程。
        - `asyncio.gather()`：并发运行多个协程，等待它们全部完成，返回一个列表，列表元素是每个协程的返回值。        

In [ ]:
import asyncio

BOUND = 10**5

# CPU密集型任务，不适用异步化
def collatz(n):
    steps = 0
    while n > 1:
        if n % 2:
            n = n * 3 + 1
        else:
            n //= 2
        steps += 1
    return steps

# 定义协程函数
async def length_counter(target):
    count = 0
    for n in range(2, BOUND):
        if collatz(n) == target:
            count += 1
        await asyncio.sleep(0)  # 主动让出控制权给事件循环，让其他协程有机会运行
    return count

# 定义协程函数
async def get_input(prompt):
    while True:
        value = input(prompt)
        # value = await ainput(prompt) # 书中通过第三方库aioconsole实现异步输入，但在Jupyter中不支持
        try:
            value = int(value)
        except ValueError:
            print("请输入一个整数")
            continue
        if value <= 0:
            print("请输入一个大于0的整数")
        else:
            return value

# 定义协程函数
async def main1():
    # 第一种调用协程函数方式：通过await关键字，等待协程函数执行完成
    target = await get_input("请输入Collatz序列的计算步数：")

    # 第二种调用协程函数方式：通过create_task()方法创建任务对象，异步调用
    counter_task = asyncio.create_task(length_counter(target))
    guess_task = asyncio.create_task(get_input("请输入你猜的Collatz序列的个数："))

    # 等待所有任务完成，获取结果
    count = await counter_task
    guess = await guess_task

    if guess == count:
        print("恭喜你猜对了！")
    else:
        print(f"很遗憾，你猜错了。实际的数是{count}")

# 定义协程函数
async def main2():
    target = await get_input("请输入Collatz序列的计算步数：")

    # 通过gather()方法并发运行多个协程，等待它们全部完成
    (count, guess) = await asyncio.gather(length_counter(target), get_input("请输入你猜的Collatz序列的个数："))

    if guess == count:
        print("恭喜你猜对了！")
    else:
        print(f"很遗憾，你猜错了。实际的数是{count}")

if __name__ == "__main__":
    await main2()
    # asyncio.run(main())  # 书中通过asyncio.run()方法启动异步事件循环，但因为Jupyter已经启动了事件循环，所以这里不能再启动

- **异步迭代器**：实现`__aiter__()`和`__anext__()`方法的迭代器，通过`async for`消费异步迭代器。
    - 异步迭代器协议：
        - `__aiter__()`：返回异步迭代器对象本身。
        - `__anext__()`：返回一个可等待对象，当await这个对象时，异步地产生下一个值；若没有更多值，则抛出StopAsyncIteration异常。
    - 创建方式：
        - 手动实现异步迭代器协议：在类中实现`__aiter__()`和`__anext__()`方法。
        - 使用异步生成器：使用`async def`配合`yield`，Python会自动为其实现异步迭代器协议。


In [ ]:
BOUND = 10**5

# 异步迭代器
class Collatz:
    def __init__(self):
        self.start = 2

    def __aiter__(self):
        return self

    async def __anext__(self):
        steps = await self.count_steps(self.start)
        self.start += 1
        if self.start == BOUND:
            raise StopAsyncIteration
        return steps

    async def count_steps(self, start_values):
        steps = 0
        n = start_values
        while n > 1:
            if n % 2:
                n = n * 3 + 1
            else:
                n //= 2
            steps += 1
        return steps

async def length_counter(target):
    count = 0
    async for steps in Collatz():  # 通过async for使用异步迭代器
        if steps == target:
            count += 1
    return count

async def get_input(prompt):
    while True:
        value = input(prompt)
        try:
            value = int(value)
        except ValueError:
            print("请输入一个整数")
            continue
        if value <= 0:
            print("请输入一个大于0的整数")
        else:
            return value

async def main():
    target = await get_input("请输入Collatz序列的计算步数：")
    (count, guess) = await asyncio.gather(length_counter(target), get_input("请输入你猜的Collatz序列的个数："))
    if guess == count:
        print("恭喜你猜对了！")
    else:
        print(f"很遗憾，你猜错了。实际的数是{count}")

if __name__ == "__main__":
    await main()

- **异步上下文管理器**：使用`async with`语句，实现了`__aenter__()`和`__aexit__()`方法，用于异步资源的获取和释放。


- **核心知识脉络**

```text
异步与并发
│
├── 1. 并发理论
│   ├── 并发 ≠ 并行（切换注意力 vs 真正同时）
│   ├── CPU-bound vs IO-bound ⭐⭐
│   │   ├── CPU-bound：并发无用，CPU是瓶颈
│   │   └── IO-bound：并发有用，等待时可做别的事
│   └── 并发的三大用途：IO等待/感知响应性/定时任务
│
├── 2. Python并发两种方式
│   ├── 线程（Threading）= 抢占式多任务 → 第17章
│   └── 异步（Asynchrony）= 协作式多任务 → 本章重点
│
├── 3. GIL（全局解释器锁）
│   ├── 单进程限单核 → 无法并行
│   └── 绕过方式：C扩展
│
├── 4. 核心概念 ⭐⭐
│   ├── 事件循环（Event Loop）—— 调度中枢
│   ├── 协程（Coroutine）—— 可暂停/恢复的函数
│   ├── async def —— 定义原生协程
│   ├── await —— 等待并让出控制权
│   └── ⚠️ await只能在async def内部使用
│
├── 5. asyncio模块 ⭐⭐
│   ├── asyncio.run(coro) —— 程序入口
│   ├── asyncio.sleep(s) —— 异步等待（vs time.sleep阻塞）
│   ├── asyncio.gather() —— 并发运行多个协程
│   ├── asyncio.create_task() —— 将协程包装为Task
│   └── asyncio.wait_for() —— 超时控制
│
├── 6. Task
│   ├── 协程的调度包装器
│   ├── 创建后立即开始运行（无需await）
│   └── await task → 获取结果
│
├── 7. 异步迭代 ⭐
│   ├── __aiter__() / __anext__() / StopAsyncIteration
│   ├── async for 循环
│   └── 每次迭代交回控制权给事件循环
│
├── 8. 异步上下文管理器
│   ├── __aenter__() / __aexit__()
│   └── async with
│
├── 9. 异步生成器
│   ├── async def + yield
│   ├── Python 3.6 (PEP 525)
│   └── 调用时返回异步生成器迭代器
│
├── 10. 高级概念（第17章预告）
│   ├── 锁/池/事件/Future
│   ├── 竞态条件/死锁
│   └── contextvars（上下文变量）
│
└── 11. 替代方案
    ├── asyncio —— 标准库，但复杂晦涩
    ├── Trio —— 更直观的第三方库
    └── 自定义事件循环
```


- **警告与提示汇总**

| 类型   | 内容                                                         |
| ------ | ------------------------------------------------------------ |
| ⚠️ 警告 | 并发**不加速执行**，因切换开销甚至可能更慢                   |
| ⚠️ 警告 | 并发对CPU-bound任务**无效**                                  |
| ⚠️ 警告 | `await`**只能在**`async def`内部使用                         |
| ⚠️ 警告 | 直接调用协程函数**不会执行**，只返回协程对象                 |
| ⚠️ 警告 | `time.sleep()`在异步代码中会**阻塞整个事件循环**，必须用`asyncio.sleep()` |
| ⚠️ 警告 | 协程完成顺序**永远无法保证**                                 |
| ⚠️ 警告 | 串行`await`多个协程=**没有并发**                             |
| ⚠️ 警告 | 协作式多任务中，协程必须**主动让出控制权**（`await`），否则会阻塞事件循环 |
| 💡 技巧 | `await asyncio.sleep(0)` 是让出控制权的惯用手法              |
| 💡 技巧 | `asyncio.gather()` 返回值顺序与传入顺序一致，与完成顺序无关  |